In [2]:
import os
import copy
import random
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models


# =========================================================
# CONFIG
# =========================================================
class CFG:
    # Kaggle path
    DATA_DIR = "/kaggle/input/datasets/yasserhessein/the-vertebrae-xray-images/xkt857dsxk-1/224/224"
    IMG_SIZE = 300
    BATCH_SIZE = 16
    EPOCHS = 30
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 0
    SEED = 42
    VAL_SIZE = 0.15
    TEST_SIZE = 0.15
    USE_WEIGHTED_SAMPLER = True
    MODEL_PATH = "/kaggle/working/best_efficientnet_b3.pth"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================================================
# LABEL MAPPING
# =========================================================
FOLDER_TO_LABEL = {
    "Normal": "Normal",
    "Scol": "Scoliosis",
    "Spond": "Spondylolisthesis"
}

LABEL_TO_INDEX = {
    "Normal": 0,
    "Scoliosis": 1,
    "Spondylolisthesis": 2
}

INDEX_TO_LABEL = {v: k for k, v in LABEL_TO_INDEX.items()}


# =========================================================
# SEED
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# DATA PREPARATION
# =========================================================
def collect_image_paths(data_dir):
    data_dir = Path(data_dir)
    samples = []

    print(f"\nUsing dataset path: {data_dir}")
    print(f"Path exists: {data_dir.exists()}")

    print("\nScanning dataset folders...")
    for folder_name, proper_label in tqdm(FOLDER_TO_LABEL.items(), desc="Classes"):
        class_dir = data_dir / folder_name

        if not class_dir.exists():
            print(f"Warning: folder not found -> {class_dir}")
            continue

        image_files = list(class_dir.glob("*"))
        print(f"{folder_name}: {len(image_files)} files found")

        for img_path in tqdm(image_files, desc=f"Reading {folder_name}", leave=False):
            if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"]:
                label_idx = LABEL_TO_INDEX[proper_label]
                samples.append((str(img_path), label_idx))

    return samples


def split_dataset(samples, val_size=0.15, test_size=0.15, seed=42):
    paths = [x[0] for x in samples]
    labels = [x[1] for x in samples]

    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        paths,
        labels,
        test_size=(val_size + test_size),
        random_state=seed,
        stratify=labels
    )

    relative_test_size = test_size / (val_size + test_size)

    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths,
        temp_labels,
        test_size=relative_test_size,
        random_state=seed,
        stratify=temp_labels
    )

    train_samples = list(zip(train_paths, train_labels))
    val_samples = list(zip(val_paths, val_labels))
    test_samples = list(zip(test_paths, test_labels))

    return train_samples, val_samples, test_samples


# =========================================================
# DATASET
# =========================================================
class SpineDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


# =========================================================
# TRANSFORMS
# =========================================================
def get_transforms(img_size=300):
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    valid_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    return train_transform, valid_transform


# =========================================================
# DATALOADERS
# =========================================================
def make_loaders(train_samples, val_samples, test_samples):
    train_tfms, valid_tfms = get_transforms(CFG.IMG_SIZE)

    train_ds = SpineDataset(train_samples, transform=train_tfms)
    val_ds = SpineDataset(val_samples, transform=valid_tfms)
    test_ds = SpineDataset(test_samples, transform=valid_tfms)

    if CFG.USE_WEIGHTED_SAMPLER:
        train_labels = [label for _, label in train_samples]
        class_count = np.bincount(train_labels)
        class_weights = 1.0 / class_count
        sample_weights = [class_weights[label] for label in train_labels]

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=CFG.BATCH_SIZE,
            sampler=sampler,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=True
        )
    else:
        train_loader = DataLoader(
            train_ds,
            batch_size=CFG.BATCH_SIZE,
            shuffle=True,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=True
        )

    val_loader = DataLoader(
        val_ds,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader


# =========================================================
# MODEL
# =========================================================
def build_model(num_classes=3):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


# =========================================================
# TRAIN ONE EPOCH
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    preds_all = []
    labels_all = []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Train]", leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

        current_loss = running_loss / len(labels_all)
        current_acc = accuracy_score(labels_all, preds_all)
        pbar.set_postfix(loss=f"{current_loss:.4f}", acc=f"{current_acc:.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(labels_all, preds_all)
    epoch_f1 = f1_score(labels_all, preds_all, average="macro")

    return epoch_loss, epoch_acc, epoch_f1


# =========================================================
# VALIDATE ONE EPOCH
# =========================================================
@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, epoch):
    model.eval()
    running_loss = 0.0
    preds_all = []
    labels_all = []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Val]", leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

        current_loss = running_loss / len(labels_all)
        current_acc = accuracy_score(labels_all, preds_all)
        pbar.set_postfix(loss=f"{current_loss:.4f}", acc=f"{current_acc:.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(labels_all, preds_all)
    epoch_f1 = f1_score(labels_all, preds_all, average="macro")

    return epoch_loss, epoch_acc, epoch_f1


# =========================================================
# TEST EVALUATION
# =========================================================
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    preds_all = []
    labels_all = []

    pbar = tqdm(loader, desc="Testing")

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.numpy())

    print("\nTest Accuracy:", accuracy_score(labels_all, preds_all))
    print("Test Macro F1:", f1_score(labels_all, preds_all, average="macro"))

    target_names = [INDEX_TO_LABEL[i] for i in range(len(INDEX_TO_LABEL))]
    print("\nClassification Report:")
    print(classification_report(labels_all, preds_all, target_names=target_names))

    print("Confusion Matrix:")
    print(confusion_matrix(labels_all, preds_all))


# =========================================================
# SINGLE IMAGE PREDICTION
# =========================================================
@torch.no_grad()
def predict_image(model, image_path, device):
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    outputs = model(image_tensor)
    probs = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probs).item()

    proper_label = INDEX_TO_LABEL[pred_idx]
    confidence = probs[pred_idx].item()

    print(f"Predicted disease: {proper_label}")
    print(f"Confidence: {confidence:.4f}")

    all_probs = {INDEX_TO_LABEL[i]: float(p) for i, p in enumerate(probs.cpu().numpy())}
    return proper_label, confidence, all_probs


# =========================================================
# MAIN
# =========================================================
def main():
    set_seed(CFG.SEED)
    print("Using device:", CFG.DEVICE)

    samples = collect_image_paths(CFG.DATA_DIR)
    print(f"\nTotal images found: {len(samples)}")

    if len(samples) == 0:
        raise ValueError("No images found. Check DATA_DIR path and folder names.")

    train_samples, val_samples, test_samples = split_dataset(
        samples,
        val_size=CFG.VAL_SIZE,
        test_size=CFG.TEST_SIZE,
        seed=CFG.SEED
    )

    print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

    train_loader, val_loader, test_loader = make_loaders(
        train_samples, val_samples, test_samples
    )

    model = build_model(num_classes=3).to(CFG.DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3
    )

    best_f1 = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    epoch_bar = tqdm(range(1, CFG.EPOCHS + 1), desc="Training Progress")

    for epoch in epoch_bar:
        train_loss, train_acc, train_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, CFG.DEVICE, epoch
        )

        val_loss, val_acc, val_f1 = validate_one_epoch(
            model, val_loader, criterion, CFG.DEVICE, epoch
        )

        scheduler.step(val_f1)

        print(
            f"\nEpoch [{epoch}/{CFG.EPOCHS}] "
            f"| Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f} "
            f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}"
        )

        epoch_bar.set_postfix(best_val_f1=f"{best_f1:.4f}", current_val_f1=f"{val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, CFG.MODEL_PATH)
            print(f"Best model saved to {CFG.MODEL_PATH}")

    print("\nLoading best model...")
    model.load_state_dict(best_weights)

    print("\nEvaluating on test set...")
    evaluate_model(model, test_loader, CFG.DEVICE)


if __name__ == "__main__":
    main()

Using device: cuda

Using dataset path: /kaggle/input/datasets/yasserhessein/the-vertebrae-xray-images/xkt857dsxk-1/224/224
Path exists: True

Scanning dataset folders...


Classes:   0%|          | 0/3 [00:00<?, ?it/s]

Normal: 71 files found


Reading Normal:   0%|          | 0/71 [00:00<?, ?it/s]

Scol: 188 files found


Reading Scol:   0%|          | 0/188 [00:00<?, ?it/s]

... (92 outputs left out)

Epoch 30 [Val]:   0%|          | 0/4 [00:00<?, ?it/s]


Epoch [30/30] | Train Loss: 0.0402 | Train Acc: 0.9958 | Train F1: 0.9958 | Val Loss: 0.0508 | Val Acc: 0.9804 | Val F1: 0.9783

Loading best model...

Evaluating on test set...


Testing:   0%|          | 0/4 [00:00<?, ?it/s]


Test Accuracy: 0.9607843137254902
Test Macro F1: 0.9493562628574072

Classification Report:
                   precision    recall  f1-score   support

           Normal       0.91      0.91      0.91        11
        Scoliosis       0.97      1.00      0.98        28
Spondylolisthesis       1.00      0.92      0.96        12

         accuracy                           0.96        51
        macro avg       0.96      0.94      0.95        51
     weighted avg       0.96      0.96      0.96        51

Confusion Matrix:
[[10  1  0]
 [ 0 28  0]
 [ 1  0 11]]


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_arch_list())